# UC2 — Gemelo Digital: EDA Patrones de Gasto por MCC

**Autor:** Brayan Ivan | UC2 — Gemelo Digital  
**Proyecto:** datamoles — Datathon DSC x Hey 2026  
**Producto:** Havi — copiloto financiero proactivo con IA

---

Este notebook explora los patrones de gasto por `categoria_mcc` del dataset de transacciones.  
El objetivo es construir las bases del **Gemelo Digital**: un perfil conductual de cada usuario a partir de cómo, cuándo y en qué gasta.

> **Salida esperada:** features candidatas para el clustering/clasificación de personas financieras en el siguiente notebook (`02_clustering_digital_twin_bi.ipynb`).

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Display options — ver más columnas y filas en los outputs
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 120)

# Tema global de seaborn
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# ──────────────────────────────────────────────
# Paths — ajustar si la estructura de carpetas cambia
# ──────────────────────────────────────────────
DATA_PATH_TRANSACCIONES = '/Users/diegodq/Downloads/dataton/dataset_transacciones/hey_transacciones.csv'
DATA_PATH_CLIENTES      = '/Users/diegodq/Downloads/dataton/dataset_transacciones/hey_clientes.csv'

print('Setup completo ✓')

## 1. Carga de Datos

In [ ]:
%%time
df_tx = pd.read_csv(DATA_PATH_TRANSACCIONES)
print(f'Transacciones cargadas: {df_tx.shape[0]:,} filas × {df_tx.shape[1]} columnas')

In [ ]:
%%time
df_cli = pd.read_csv(DATA_PATH_CLIENTES)
print(f'Clientes cargados: {df_cli.shape[0]:,} filas × {df_cli.shape[1]} columnas')

In [ ]:
# Vista rápida de ambos datasets
print('── Transacciones ──')
display(df_tx.head(3))
print('\n── Clientes ──')
display(df_cli.head(3))

In [ ]:
# Info general — tipos de datos y nulos
print('── Transacciones info ──')
df_tx.info()
print('\n── Clientes info ──')
df_cli.info()

## 2. Exploración de `categoria_mcc`

### ¿Qué son los códigos MCC y por qué importan?

**MCC (Merchant Category Code)** es un código de 4 dígitos que las redes de pago (Visa, Mastercard) asignan a cada comercio según el tipo de negocio. En nuestro dataset, `categoria_mcc` es la versión etiquetada/agrupada de ese código.

Para el **Gemelo Digital**, los MCCs son el insumo más rico: revelan *qué tipo de vida lleva el usuario*. Alguien que gasta frecuentemente en categorías como `restaurantes`, `entretenimiento` y `viajes` tiene un perfil conductual completamente distinto a alguien concentrado en `supermercados` y `farmacia`.

Son la base para features como diversidad de categorías, concentración de gasto y clasificación de persona financiera.

In [ ]:
# Detección dinámica de la columna MCC — no asumir nombre exacto
COL_MCC = next(
    (c for c in df_tx.columns if 'mcc' in c.lower() or 'categoria' in c.lower()),
    None
)

if COL_MCC is None:
    raise ValueError('No se encontró columna MCC/categoría. Revisar columnas:', df_tx.columns.tolist())

print(f'Columna MCC detectada: "{COL_MCC}"')
print(f'Valores únicos: {df_tx[COL_MCC].nunique()}')
print()
display(df_tx[COL_MCC].value_counts().head(20).rename('conteo_transacciones').to_frame())

In [ ]:
# Detección dinámica de columna de monto
COL_MONTO = next(
    (c for c in df_tx.columns if 'monto' in c.lower() or 'amount' in c.lower() or 'importe' in c.lower()),
    None
)
print(f'Columna monto detectada: "{COL_MONTO}"')

In [ ]:
# ── Top 20 categorías por número de transacciones ──
top20_count = df_tx[COL_MCC].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=top20_count.values, y=top20_count.index, ax=ax, palette='Blues_r')
ax.set_title('Top 20 Categorías MCC — por Número de Transacciones', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de Transacciones', fontsize=11)
ax.set_ylabel('Categoría MCC', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 20 categorías por monto total ──
# Separar cargos (negativos o tipo='cargo') de abonos para no mezclar señales
if COL_MONTO:
    top20_monto = (
        df_tx.groupby(COL_MCC)[COL_MONTO]
        .sum()
        .abs()               # valor absoluto para comparar magnitudes
        .sort_values(ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(x=top20_monto.values, y=top20_monto.index, ax=ax, palette='Oranges_r')
    ax.set_title('Top 20 Categorías MCC — por Monto Total Transaccionado', fontsize=14, fontweight='bold')
    ax.set_xlabel('Monto Total (valor absoluto)', fontsize=11)
    ax.set_ylabel('Categoría MCC', fontsize=11)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    plt.tight_layout()
    plt.show()
else:
    print('No se detectó columna de monto. Saltando gráfico.')

## 3. Distribución de Montos por Categoría

In [ ]:
if COL_MONTO:
    # Top 10 categorías por frecuencia para mantener el gráfico legible
    top10_cats = df_tx[COL_MCC].value_counts().head(10).index.tolist()
    df_top10 = df_tx[df_tx[COL_MCC].isin(top10_cats)].copy()

    # Usar valor absoluto del monto — en algunos datasets cargos son negativos
    df_top10['monto_abs'] = df_top10[COL_MONTO].abs()

    # ── Boxplot con escala logarítmica ──
    # Log scale para manejar la distribución leptocúrtica típica de transacciones
    fig, ax = plt.subplots(figsize=(14, 6))
    order = df_top10.groupby(COL_MCC)['monto_abs'].median().sort_values(ascending=False).index
    sns.boxplot(data=df_top10, x=COL_MCC, y='monto_abs', order=order, ax=ax,
                palette='Set2', flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax.set_yscale('log')
    ax.set_title('Distribución de Montos por Categoría MCC (Top 10) — Escala Log', fontsize=13, fontweight='bold')
    ax.set_xlabel('Categoría MCC', fontsize=11)
    ax.set_ylabel('Monto (log scale)', fontsize=11)
    ax.tick_params(axis='x', rotation=40)
    plt.tight_layout()
    plt.show()

In [ ]:
if COL_MONTO:
    # ── Violin plot — muestra la densidad completa, no solo los cuartiles ──
    # Más informativo que el boxplot para distribuciones multimodales
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.violinplot(data=df_top10, x=COL_MCC, y='monto_abs', order=order, ax=ax,
                   palette='Set2', scale='width', inner='quartile', cut=0)
    ax.set_yscale('log')
    ax.set_title('Distribución de Montos por Categoría MCC (Violin) — Escala Log', fontsize=13, fontweight='bold')
    ax.set_xlabel('Categoría MCC', fontsize=11)
    ax.set_ylabel('Monto (log scale)', fontsize=11)
    ax.tick_params(axis='x', rotation=40)
    plt.tight_layout()
    plt.show()

In [ ]:
if COL_MONTO:
    # ── Categorías de alta varianza — señales para anomaly detection y profiling ──
    # CV (coeficiente de variación) = std / mean — normaliza la dispersión
    stats_por_cat = (
        df_tx.groupby(COL_MCC)[COL_MONTO]
        .agg(['mean', 'std', 'count'])
        .assign(cv=lambda x: x['std'] / x['mean'].abs())
        .sort_values('cv', ascending=False)
    )
    print('Categorías con mayor variabilidad relativa (CV = std/mean):')
    display(stats_por_cat.head(15).style.format({'mean': '${:,.0f}', 'std': '${:,.0f}', 'cv': '{:.2f}'}))

## 4. Patrones Temporales por Categoría

### ¿Qué revelan los patrones temporales?

El **cuándo** gasta un usuario es tan informativo como el **qué**. Un usuario que gasta más en fin de semana en entretenimiento tiene un perfil distinto a uno que gasta en días hábiles en servicios profesionales. Estos patrones temporales alimentan features como `feat_weekend_spend_ratio` y permiten detectar comportamientos estacionales (ej. gastos navideños elevados en diciembre).

In [ ]:
# Detección dinámica de columna de fecha
COL_FECHA = next(
    (c for c in df_tx.columns if 'fecha' in c.lower() or 'date' in c.lower() or 'tiempo' in c.lower()),
    None
)
print(f'Columna fecha detectada: "{COL_FECHA}"')

if COL_FECHA:
    df_tx[COL_FECHA] = pd.to_datetime(df_tx[COL_FECHA], errors='coerce')
    df_tx['mes']         = df_tx[COL_FECHA].dt.to_period('M')
    df_tx['dia_semana']  = df_tx[COL_FECHA].dt.dayofweek   # 0=Lunes, 6=Domingo
    df_tx['es_finde']    = df_tx['dia_semana'].isin([5, 6]).astype(int)
    print(f'Rango de fechas: {df_tx[COL_FECHA].min()} → {df_tx[COL_FECHA].max()}')
    print(f'Nulos en fecha: {df_tx[COL_FECHA].isna().sum():,}')
else:
    print('No se detectó columna de fecha. Salteando análisis temporal.')

In [ ]:
if COL_FECHA and COL_MONTO:
    top5_cats = df_tx[COL_MCC].value_counts().head(5).index.tolist()

    # Gasto mensual por categoría — top 5
    monthly_cat = (
        df_tx[df_tx[COL_MCC].isin(top5_cats)]
        .groupby(['mes', COL_MCC])[COL_MONTO]
        .sum().abs()
        .reset_index()
    )
    monthly_cat['mes_str'] = monthly_cat['mes'].astype(str)

    fig, ax = plt.subplots(figsize=(14, 5))
    for cat in top5_cats:
        subset = monthly_cat[monthly_cat[COL_MCC] == cat].sort_values('mes_str')
        ax.plot(subset['mes_str'], subset[COL_MONTO], marker='o', label=cat, linewidth=2)

    ax.set_title('Tendencia Mensual de Gasto — Top 5 Categorías MCC', fontsize=13, fontweight='bold')
    ax.set_xlabel('Mes', fontsize=11)
    ax.set_ylabel('Monto Total', fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Categoría', bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [ ]:
if COL_FECHA and COL_MONTO:
    top10_cats = df_tx[COL_MCC].value_counts().head(10).index.tolist()
    DIAS = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

    # Heatmap: categoría × día de semana — gasto promedio por celda
    heatmap_data = (
        df_tx[df_tx[COL_MCC].isin(top10_cats)]
        .groupby([COL_MCC, 'dia_semana'])[COL_MONTO]
        .mean().abs()
        .unstack(fill_value=0)
    )
    heatmap_data.columns = [DIAS[i] for i in heatmap_data.columns]

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.4,
                cbar_kws={'label': 'Monto Promedio'}, ax=ax)
    ax.set_title('Monto Promedio por Categoría MCC y Día de Semana', fontsize=13, fontweight='bold')
    ax.set_xlabel('Día de la Semana', fontsize=11)
    ax.set_ylabel('Categoría MCC', fontsize=11)
    plt.tight_layout()
    plt.show()

## 5. Comportamiento por Usuario

### ¿Por qué analizar el comportamiento individual?

La vista agregada (sección anterior) nos dice qué categorías son populares *en general*, pero para el **Gemelo Digital** necesitamos la perspectiva del usuario individual. La diversidad de categorías, el ticket promedio y la concentración de gasto son los bloques fundamentales para clasificar personas financieras (ej.: `ahorrador_cauteloso`, `consumidor_activo`, `viajero_frecuente`).

In [ ]:
# Detección dinámica de columna user_id
COL_USER = next(
    (c for c in df_tx.columns if 'user' in c.lower() or 'cliente' in c.lower() or 'id' in c.lower()),
    df_tx.columns[0]  # fallback a primera columna
)
print(f'Columna user_id detectada: "{COL_USER}"')
print(f'Usuarios únicos en transacciones: {df_tx[COL_USER].nunique():,}')

In [ ]:
if COL_MONTO:
    # Pivot: usuario × categoría = suma de montos
    # Esta tabla es la base para clustering en el siguiente notebook
    pivot_user_cat = (
        df_tx.groupby([COL_USER, COL_MCC])[COL_MONTO]
        .sum().abs()
        .unstack(fill_value=0)
    )
    print(f'Pivot usuario × categoría: {pivot_user_cat.shape}')
    display(pivot_user_cat.head(5))

In [ ]:
# Diversidad de categorías por usuario
cat_diversity = df_tx.groupby(COL_USER)[COL_MCC].nunique().rename('n_categorias')

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(cat_diversity, bins=30, kde=True, ax=ax, color='steelblue')
ax.set_title('Distribución de Diversidad de Categorías por Usuario', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Categorías MCC únicas', fontsize=11)
ax.set_ylabel('Cantidad de Usuarios', fontsize=11)
ax.axvline(cat_diversity.median(), color='crimson', linestyle='--', label=f'Mediana: {cat_diversity.median():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

print(cat_diversity.describe())

In [ ]:
# Top categorías por cobertura de usuarios — ¿en cuántos usuarios aparece cada categoría?
users_per_cat = (
    df_tx.groupby(COL_MCC)[COL_USER]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
    .rename('usuarios_unicos')
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=users_per_cat.values, y=users_per_cat.index, ax=ax, palette='viridis')
ax.set_title('Top 20 Categorías MCC por Cantidad de Usuarios Únicos', fontsize=13, fontweight='bold')
ax.set_xlabel('Usuarios Únicos', fontsize=11)
ax.set_ylabel('Categoría MCC', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
# Usuarios mono-categoría vs diversificados
mono_cat     = (cat_diversity == 1).sum()
diversified  = (cat_diversity >= 5).sum()
total_users  = cat_diversity.shape[0]

print(f'Total usuarios:                  {total_users:,}')
print(f'Mono-categoría (solo 1):         {mono_cat:,}  ({mono_cat/total_users*100:.1f}%)')
print(f'Diversificados (5+ categorías):  {diversified:,}  ({diversified/total_users*100:.1f}%)')

## 6. Segmentación Preliminar por Gasto

### Hipótesis sobre categorías y tipos de persona financiera

Antes de correr el clustering formal (siguiente notebook), podemos proponer hipótesis basadas en lo observado:

- **Alto gasto + alta diversidad** → perfil de consumidor activo / lifestyle
- **Alto gasto + baja diversidad** (concentrado en 1-2 categorías) → especialista / hábito específico (ej. gasolina + restaurantes)
- **Bajo gasto + alta diversidad** → usuario esporádico / exploratorio
- **Bajo gasto + baja diversidad** → usuario básico / potencial inactivo

La combinación de `gasto_total`, `n_categorias` y `categoria_top` es suficiente para una primera segmentación exploratoria.

In [ ]:
if COL_MONTO:
    # Features base por usuario — serán las inputs del clustering
    user_stats = df_tx.groupby(COL_USER).agg(
        gasto_total   = (COL_MONTO, lambda x: x.abs().sum()),
        ticket_avg    = (COL_MONTO, lambda x: x.abs().mean()),
        n_transacc    = (COL_MONTO, 'count'),
        n_categorias  = (COL_MCC,   'nunique'),
        top_categoria = (COL_MCC,   lambda x: x.value_counts().idxmax()),
    ).reset_index()

    # Segmentación simple por cuartiles de gasto total
    user_stats['segmento_gasto'] = pd.qcut(
        user_stats['gasto_total'],
        q=4,
        labels=['bajo', 'medio-bajo', 'medio-alto', 'alto']
    )

    print(f'Usuarios con features calculadas: {user_stats.shape[0]:,}')
    display(user_stats.head(5))

In [ ]:
if COL_MONTO:
    # Distribución de usuarios por segmento
    seg_counts = user_stats['segmento_gasto'].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(x=seg_counts.index.astype(str), y=seg_counts.values, ax=ax,
                palette=['#d4e6f1', '#7fb3d3', '#2e86c1', '#1a5276'])
    ax.set_title('Distribución de Usuarios por Segmento de Gasto (Cuartiles)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Segmento', fontsize=11)
    ax.set_ylabel('Cantidad de Usuarios', fontsize=11)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# Crosstab con segmento del dataset de clientes (si está disponible)
COL_SEGMENTO = next(
    (c for c in df_cli.columns if 'segmento' in c.lower() or 'segment' in c.lower()),
    None
)
COL_USER_CLI = next(
    (c for c in df_cli.columns if 'user' in c.lower() or 'cliente' in c.lower() or 'id' in c.lower()),
    df_cli.columns[0]
)

if COL_SEGMENTO and COL_MONTO:
    df_merged = user_stats.merge(
        df_cli[[COL_USER_CLI, COL_SEGMENTO]].rename(columns={COL_USER_CLI: COL_USER}),
        on=COL_USER, how='left'
    )
    crosstab = pd.crosstab(df_merged['segmento_gasto'], df_merged[COL_SEGMENTO], margins=True)
    print(f'Crosstab: segmento de gasto (calculado) vs {COL_SEGMENTO} (dataset clientes)')
    display(crosstab)
else:
    print(f'Columna segmento en clientes no encontrada ({COL_SEGMENTO=}). Saltando crosstab.')

## 7. Features Candidatas para UC2

### Features derivadas de este análisis

Estas features son los inputs recomendados para el clustering/clasificación del siguiente notebook (`02_clustering_digital_twin_bi.ipynb`):

| Feature | Descripción | Derivada de |
|---|---|---|
| `feat_top_category` | Categoría MCC con mayor frecuencia por usuario | `value_counts` por user_id |
| `feat_category_diversity` | Número de categorías únicas por usuario | `nunique` de MCC por user_id |
| `feat_avg_ticket_by_category` | Ticket promedio por categoría para cada usuario | `mean` de monto, agrupado por user_id + MCC |
| `feat_spend_concentration` | % del gasto total en la categoría principal (Herfindahl-like) | `top_cat_spend / total_spend` |
| `feat_weekend_spend_ratio` | % del gasto total en fines de semana | `finde_spend / total_spend` |
| `feat_gasto_total` | Gasto total en el período | suma de montos absolutos |
| `feat_ticket_avg` | Ticket promedio del usuario | media de montos absolutos |
| `feat_n_transacciones` | Frecuencia de uso (proxy de actividad) | count de transacciones |

> **Nota:** Estas features deben normalizarse (StandardScaler o MinMaxScaler) antes de entrar al modelo de clustering. Las categóricas (`feat_top_category`) requieren encoding o tratamiento separado.

In [ ]:
if COL_MONTO:
    # Calcular feat_spend_concentration: % del gasto en la categoría top
    top_cat_spend = (
        df_tx.groupby([COL_USER, COL_MCC])[COL_MONTO]
        .sum().abs()
        .reset_index()
        .sort_values(COL_MONTO, ascending=False)
        .groupby(COL_USER)
        .first()[COL_MONTO]
        .rename('top_cat_spend')
    )

    user_feats = user_stats.set_index(COL_USER).copy()
    user_feats['feat_spend_concentration'] = (
        top_cat_spend / user_feats['gasto_total']
    ).clip(0, 1)  # por si hay redondeos

    # feat_weekend_spend_ratio (solo si tenemos fecha)
    if COL_FECHA:
        finde_spend = (
            df_tx[df_tx['es_finde'] == 1]
            .groupby(COL_USER)[COL_MONTO]
            .sum().abs()
            .rename('finde_spend')
        )
        user_feats['feat_weekend_spend_ratio'] = (
            finde_spend / user_feats['gasto_total']
        ).fillna(0).clip(0, 1)

    # Renombrar para convención feat_*
    user_feats = user_feats.rename(columns={
        'gasto_total':   'feat_gasto_total',
        'ticket_avg':    'feat_ticket_avg',
        'n_transacc':    'feat_n_transacciones',
        'n_categorias':  'feat_category_diversity',
        'top_categoria': 'feat_top_category',
    })

    print(f'Features candidatas calculadas: {user_feats.shape}')
    display(user_feats.drop(columns=['segmento_gasto'], errors='ignore').head(5))
    display(user_feats.drop(columns=['segmento_gasto', 'feat_top_category'], errors='ignore').describe())

## 8. Resumen Ejecutivo

### Resumen Ejecutivo — completar después de correr el notebook

**Fecha de análisis:** _(completar)_  
**Dataset:** `hey_transacciones.csv`  
**Autor:** Brayan Ivan

---

#### Categorías MCC principales encontradas
_(completar con las top 5 por frecuencia y por monto)_

#### Patrones de comportamiento identificados
- Diversidad promedio de categorías por usuario: _(completar)_
- % de usuarios mono-categoría: _(completar)_
- Categoría con mayor ticket promedio: _(completar)_
- Patrón temporal relevante: _(completar)_

#### Problemas de calidad de datos
- [ ] Nulos en columna de fecha: _(completar)_
- [ ] Outliers en montos: _(completar)_
- [ ] Categorías con muy baja frecuencia (posible agrupación necesaria): _(completar)_

#### Features recomendadas para Feature Engineering
1. `feat_top_category` — alta señal para clustering
2. `feat_category_diversity` — discriminador clave entre personas
3. `feat_spend_concentration` — proxy de especialización vs. diversificación
4. `feat_weekend_spend_ratio` — diferencia perfiles de lifestyle
5. `feat_ticket_avg` — proxy de poder adquisitivo

#### Próximos pasos
- Construir pipeline de Feature Engineering completo en `02_clustering_digital_twin_bi.ipynb`
- Evaluar si se necesita reducción dimensional (PCA/UMAP) antes del clustering
- Definir número de clusters con método del codo + silhouette score
- Cruzar clusters con dataset de clientes para validar coherencia demográfica